In [2]:
%matplotlib qt5

import hyperspy.api as hs
import pyxem as pxm
import numpy as np
import orix
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import matplotlib as mpl
import pickle

from pathlib import Path
from diffpy.structure import Atom, Lattice, Structure
from diffsims.generators.simulation_generator import SimulationGenerator
from dataclasses import dataclass
from sklearn.decomposition import PCA

from orix.quaternion import symmetry, OrientationRegion
from orix.vector import Vector3d
from orix.plot import IPFColorKeyTSL

c:\Users\denis\hyperspy-bundle_latest\Lib\site-packages\hyperspy\misc\utils.py:72: VisibleDeprecationWarning: `_get_block_pattern` has moved to `hyperspy.misc.dask_utils`. It is for internal use only and may be removed in the future.
  warnings.warn(


In [4]:
path = Path("D:\\Datasets\\silver_tilt_series_2\\20250225_175525\\SPED5.mib")
data = hs.load(path)

In [7]:
shifts = data.get_direct_beam_position(method="blur", sigma=5, half_square_width=10)
linear_shifts = shifts.get_linear_plane()
data_centered = data.center_direct_beam(shifts=linear_shifts, inplace=False)

  0%|          | 0/197 [00:00<?, ?it/s]

  0%|          | 0/97 [00:00<?, ?it/s]

In [12]:
ny, nx = data_centered.axes_manager.signal_shape
centers = (ny/2 - 0.5, nx/2 - 0.5)
data_centered.calibration(center=centers)

In [13]:
# Define the crystal structure for aluminum (Al)
a = 4.08
atoms = [Atom('Al', [0, 0, 0]), Atom('Al', [0.5, 0.5, 0]), Atom('Al', [0.5, 0, 0.5]), Atom('Al', [0, 0.5, 0.5])]
lattice = Lattice(a, a, a, 90, 90, 90)
phase = orix.crystal_map.Phase(name='Al', space_group=225, structure=Structure(atoms, lattice))

angular_resolution = 0.5
grid = orix.sampling.get_sample_reduced_fundamental(angular_resolution, point_group=phase.point_group)
orientations = orix.quaternion.Orientation(grid, symmetry=phase.point_group)

simgen = SimulationGenerator(precession_angle=1., minimum_intensity=1e-5)
simulations = simgen.calculate_diffraction2d(phase=phase, rotation=grid, reciprocal_radius=1.5, with_direct_beam=False, max_excitation_error=0.01)

# Create IPF color keys for x, y, and z directions
key_x = IPFColorKeyTSL(phase.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(phase.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(phase.point_group, Vector3d.zvector())

In [15]:
data_centered.change_dtype("float32")
polar = data_centered.get_azimuthal_integral2d(npt=256)

  0%|          | 0/73 [00:00<?, ?it/s]

In [16]:
result = polar.get_orientation(simulations, n_best=grid.size, frac_keep=0.4)

  0%|          | 0/290 [00:00<?, ?it/s]

WARNING | Hyperspy | The function you applied does not take into account the difference of scales in-between axes. (hyperspy.signal:5597)
WARNING | Hyperspy | The function you applied does not take into account the difference of units in-between axes. (hyperspy.signal:5602)


  0%|          | 0/33 [00:00<?, ?it/s]

In [19]:
data_centered.plot(cmap="viridis", norm="symlog")
data_centered.add_marker(result.to_markers(annotate=True))

In [21]:
data_centered.axes_manager

Navigation axis name,size,index,offset,scale,units
,200,28,0.0,1.0,
,100,59,0.0,1.0,
Signal axis name,size,,offset,scale,units
,256,,-1.733718727187314,0.013597793938724031,1/Å
,256,,-1.733718727187314,0.013597793938724031,1/Å
